# Personal Styling Assistant

This notebook implements a minimal agent that picks outfits from a made up wardrobe by calling tools. The same model is evaluated untouched and then after LoRA fine tuning, on the same held out test set.

| | |
|---|---|
| Model | `Qwen2.5-0.5B-Instruct` (fp16, greedy decoding) |
| Tools | `get_venue_info`, `get_weather`, `search_wardrobe`, `check_color_harmony` |
| Data | 1200 train / 100 val / 100 test, generated from templates, test set uses held out phrasings |
| LoRA | rank 16, alpha = 2 × rank, all attention + MLP projections, loss on assistant turns only |
| Seed | 42 (fixed in `config.yaml`) |

**Runtime on a free T4:** Sections 1–7 and 9–10 takes about 40 minutes of GPU time. The optional rank 4 run (section 8) adds about 30 minutes.
All outputs are saved to Google Drive. After a disconnect, run all cells again. Finished steps are skipped and training resumes from its last checkpoint.

The code is in the repo: https://github.com/sasindipeiris/lora-styling-agent

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/lora-styling-agent'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/sasindipeiris/lora-styling-agent.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt

In [ ]:
# Keep outputs (adapters, checkpoints, predictions, metrics) on Drive so a disconnect is not fatal.
DRIVE_OUT = '/content/drive/MyDrive/styling-agent/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if not os.path.exists('outputs'):
    os.symlink(DRIVE_OUT, 'outputs')
print('outputs ->', os.path.realpath('outputs'))

In [ ]:
# T4 setting from config.yaml: batch 1 x grad_accum 8 (effective batch of 8).
import re
from pathlib import Path
cfg = Path('config.yaml').read_text()
cfg = re.sub(r'batch_size: \d+', 'batch_size: 1', cfg)
cfg = re.sub(r'grad_accum: \d+', 'grad_accum: 8', cfg)
Path('config.yaml').write_text(cfg)

import yaml
CFG = yaml.safe_load(cfg)
print('seed:', CFG['seed'], '| model:', CFG['model'])
print('data:', CFG['data'])
print('lora:', CFG['lora'])
print('train:', CFG['train'])

## 2. Data

`build_db` creates the mock database. `split` then generates the requests and runs the real tools to produce each correct trace. Generation is seeded.
- **Kinds of request:** `no_tool` (no tool needed), `single_tool` (one lookup) and `full_outfit` (5 to 8 chained calls).
- **Held out test set:** test requests use different templates and some wording never seen in training (for example "night" instead of "in the evening"). Test outfit requests also use venue/event/day/time combinations that never appear in training.

In [ ]:
!python -m src.dataset.build_db
!python -m src.dataset.split

In [ ]:
import json
from collections import Counter
from src.utils import read_jsonl
data = {s: read_jsonl(f'data/{s}.jsonl') for s in ('train', 'val', 'test')}
for s, rows in data.items():
    print(f'{s:5s} {len(rows):5d}', dict(Counter(r['kind'] for r in rows)))

In [ ]:
def show_trace(rec):
    print('USER:', rec['user'])
    for m in rec['messages'][1:]:
        if m.get('tool_calls'):
            f = m['tool_calls'][0]['function']
            print(f"  CALL{'' if m.get('train', True) else ' (masked)'}: {f['name']}({json.dumps(f['arguments'])})")
        elif m['role'] == 'tool':
            print('  RESULT:', m['content'][:150])
        else:
            print('ANSWER:', m['content'])

show_trace(next(r for r in data['train'] if r['kind'] == 'full_outfit'))
print()
show_trace(next(r for r in data['train'] if r['kind'] == 'no_tool'))
print('\nTrain phrasing:', next(r['user'] for r in data['train'] if r['kind'] == 'full_outfit'))
print('Test phrasing: ', next(r['user'] for r in data['test'] if r['kind'] == 'full_outfit'))

## 3. Tools

These are plain Python functions over the mock database. If a call fails, the agent gets back an `{"error": ...}` message that lists the valid options so the model can see the problem and correct its call.

In [ ]:
from src.constants import TOOL_SCHEMAS
from src.tools import call_tool
for s in TOOL_SCHEMAS:
    f = s['function']
    print(f"{f['name']}({', '.join(f['parameters']['properties'])}): {f['description']}")
print()
for name, args in [('get_venue_info', {'venue': 'gfh', 'event_type': 'dinner'}),
                   ('get_weather', {'city': 'Colombo', 'day': 'friday', 'time_of_day': 'evening'}),
                   ('search_wardrobe', {'category': 'shoes', 'min_formality': 4, 'rules': ['closed_shoes']}),
                   ('check_color_harmony', {'item_ids': ['D05', 'S02']}),
                   ('get_weather', {'city': 'Colombo', 'day': 'friday', 'time_of_day': 'night'})]:
    print(f'{name}({args})\n  -> {call_tool(name, args)}')

## 4. Agent loop

`src/agent.py` repeats **generate → parse `<tool_call>` → execute → feed the result back**. It stops when the model replies in plain text (or writes a call that can't be parsed), or after 10 steps.

In [ ]:
import inspect
from src import agent
print(inspect.getsource(agent.run_agent))

In [ ]:
import gc, torch
from src.agent import run_agent
from src.chat import show
from src.utils import load_model

DEMO = ['What should I wear to a dinner at Galle Face Hotel on Friday in the evening?',
        'How do I wash a silk saree?']

def demo(tag):
    model, tok = load_model(tag)
    for q in DEMO:
        print(f'\nUSER: {q}')
        out = run_agent(model, tok, [{'role': 'user', 'content': q}], on_step=show)
        print(f"AGENT [{out['stopped']}]: {out['final']}")
    del model; gc.collect(); torch.cuda.empty_cache()

demo('zeroshot')   # untouched base model

## 5. Baseline Evaluation

The base model runs on all 100 test requests. The scores are:
- **call validity:** every tool call parses
- **call accuracy:** right tool and right arguments, judged by whether the call returns the same result as the correct call
- **abstention:** no tool call when none is needed
- **task success:** the final answer is correct end to end. For outfits, this means the chosen items pass `check_outfit`.

In [ ]:
import pandas as pd

def evaluate(tag):
    if not os.path.exists(f'outputs/{tag}/metrics.json'):
        !python -m src.evaluate --tag {tag}
    else:
        print(f'{tag}: metrics already on Drive, skipping')

def metrics_table(tags):
    rows = {}
    for t in tags:
        p = f'outputs/{t}/metrics.json'
        if os.path.exists(p):
            m = json.load(open(p))
            rows[t] = {k: m[k] for k in ('call_validity', 'call_accuracy', 'abstention', 'task_success')} | \
                      {f'success {k}': v for k, v in m['task_success_by_kind'].items()} | {'avg steps': m['avg_steps']}
    return pd.DataFrame(rows).T

evaluate('zeroshot')
metrics_table(['zeroshot'])

## 6. LoRA finetuning (rank 16)

`src/train.py` trains with **the loss on assistant turns only**: the system prompt, user message and tool results get label −100, so they do not count. Failed calls that are kept as error recovery examples are masked too.
- **Precision:** fp16, since a T4 has no bf16.
- **Settings:** learning rate 2e-4 with a cosine schedule, 2 epochs, effective batch size 8.
- **Checkpoints:** saved to Drive every 50 steps. If the session disconnects, rerunning this cell resumes from the latest one.

In [ ]:
def train(rank):
    if not os.path.exists(f'outputs/lora_r{rank}/adapter'):
        !python -m src.train --rank {rank}
    else:
        print(f'lora_r{rank}: adapter already on Drive, skipping')

train(16)

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(tag):
    log = json.load(open(f'outputs/{tag}/train_log.json'))
    tr = [(x['step'], x['loss']) for x in log if 'loss' in x]
    ev = [(x['step'], x['eval_loss']) for x in log if 'eval_loss' in x]
    plt.plot(*zip(*tr), label='train')
    plt.plot(*zip(*ev), 'o-', label='val')
    plt.xlabel('step'); plt.ylabel('loss'); plt.yscale('log'); plt.title(tag); plt.legend(); plt.show()

plot_loss('lora_r16')

## 7. LoRA evaluation 

In [ ]:
evaluate('lora_r16')
metrics_table(['zeroshot', 'lora_r16'])

In [ ]:
demo('lora_r16')   # same two requests as in section 4

## 8. (Optional) LoRA rank 4

Run this only if there is session time left (about 30 minutes on a T4). It shows how much a smaller adapter loses.

In [ ]:
RUN_RANK_4 = False
if RUN_RANK_4:
    train(4)
    plot_loss('lora_r4')
    evaluate('lora_r4')

## 9. Results: baseline vs. LoRA

In [ ]:
TAGS = [t for t in ('zeroshot', 'lora_r16', 'lora_r4') if os.path.exists(f'outputs/{t}/metrics.json')]
metrics_table(TAGS)

## 10. Failure analysis

Failures are grouped into categories: wrong format, wrong tool, wrong argument, unnecessary call, ignored result, incomplete chain, hallucinated item and constraint violated. The cells below show:
- how many failures each run has in each category
- what LoRA fixed, did not fix and introduced compared with the baseline
- at least 10 failed requests per model

In [ ]:
preds = {t: read_jsonl(f'outputs/{t}/predictions.jsonl') for t in TAGS}
pd.DataFrame({t: Counter(r['failure'] for r in rows if r['failure']) for t, rows in preds.items()}).fillna(0).astype(int)

In [ ]:
# Which baseline failures LoRA fixed or not, and which new failures it introduced
base = {r['id']: r for r in preds['zeroshot']}
for t in TAGS[1:]:
    fixed, still, new = Counter(), Counter(), Counter()
    for r in preds[t]:
        b = base[r['id']]
        if not b['success'] and r['success']:
            fixed[b['failure']] += 1
        elif not b['success']:
            still[r['failure']] += 1
        elif not r['success']:
            new[r['failure']] += 1
    print(f'{t}:\n  fixed (by baseline category): {dict(fixed)}\n'
          f'  still failing (now as):       {dict(still)}\n  introduced:                   {dict(new)}\n')

In [ ]:
def show_failures(tag, n=10):
    print(f'=== {tag} ===')
    for r in [r for r in preds[tag] if not r['success']][:n]:
        last = r['final'] or r['steps'][-1]['raw']
        print(f"[{r['id']}] {r['failure']} {r.get('outfit_failed') or ''}\n  user:  {r['user']}\n  agent: {last[:200]!r}")
    print()

for t in TAGS:
    show_failures(t)

In [ ]:
# Recurring failure: the test-only word 'night' (training only says 'in the evening')
for t in TAGS:
    tool = [r for r in preds[t] if r['kind'] != 'no_tool']
    night = [r for r in tool if 'night' in r['user']]
    other = [r for r in tool if 'night' not in r['user']]
    print(f"{t:9s} 'night' requests: {sum(r['success'] for r in night)}/{len(night)}   "
          f"other tool requests: {sum(r['success'] for r in other)}/{len(other)}")